# Regression Tutorial Part 3: Logistic Regression

Welcome to the bridge between regression and classification. **Logistic Regression** is the foundational algorithm for binary classification. 

Instead of predicting the exact outcome directly, Logistic Regression predicts the **probability** that a given instance belongs to the default class (Class 1).

### The Mathematical Formulation:
1. First, we calculate a standard linear combination (just like Linear Regression):
   $$z = \beta_0 + \beta_1x_1 + \dots + \beta_nx_n$$
2. Second, we pass $z$ through the **Sigmoid Activation Function** to squash it between 0 and 1:
   $$\hat{y} = \frac{1}{1 + e^{-z}}$$

Where:
* $z$ is the linear output (log-odds).
* $\hat{y}$ is the predicted probability of belonging to Class 1.
* $e$ is Euler's number.

### Our Analytical Pipeline:
1. **Environment Setup:** Importing data science libraries.
2. **Data Generation:** Synthesizing a dataset of "Hours Studied" vs "Exam Pass/Fail".
3. **Model Training:** Fitting the logistic curve using Maximum Likelihood Estimation.
4. **Evaluation:** Analyzing performance using the Confusion Matrix and Accuracy.

In [ ]:
# Cell 2: Environment Setup and Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Core Scikit-Learn modules for Classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.model_selection import train_test_split

# Set seeds and styling for analytical consistency
np.random.seed(42)
sns.set_theme(style='whitegrid')
print("Libraries imported successfully!")

## Step 1: Synthetic Data Generation

We will generate a dataset simulating 200 students. Our independent variable ($X$) is **Hours Studied**, and our dependent variable ($y$) is a binary outcome: **0 (Failed)** or **1 (Passed)**.

We will inject realistic variance—some students who study a lot might still fail, and some who barely study might guess their way to a passing grade.

In [ ]:
# Cell 4: Generating the Analytical Dataset

n_students = 200

# 1. Generate Input Feature: Hours Studied (between 0 and 10 hours)
X_hours = np.random.uniform(0, 10, n_students).reshape(-1, 1)

# 2. Define the hidden mathematical rule for passing
# Let's say the "tipping point" is around 5 hours. 
# We calculate a logit (z) and pass it through a sigmoid to get a true probability.
true_intercept = -4.0
true_slope = 0.8
z = true_intercept + (true_slope * X_hours.flatten())

# The true probability of passing for each student
p_pass = 1 / (1 + np.exp(-z))

# 3. Generate actual outcomes (0 or 1) based on their individual probabilities using binomial distribution
y_pass = np.random.binomial(1, p_pass)

# Split into Training (80%) and Testing (20%)
X_train, X_test, y_train, y_test = train_test_split(X_hours, y_pass, test_size=0.2, random_state=42)

# Visualize the training data
plt.figure(figsize=(8, 5))
plt.scatter(X_train[y_train==0], y_train[y_train==0], alpha=0.6, label='Failed (0)')
plt.scatter(X_train[y_train==1], y_train[y_train==1], alpha=0.6, label='Passed (1)')
plt.title('Training Data: Hours Studied vs. Pass/Fail', fontsize=14)
plt.xlabel('Hours Studied')
plt.ylabel('Exam Outcome (Binary)')
plt.yticks([0, 1])
plt.legend(loc='center right')
plt.show()

## Step 2: Model Training and The Decision Boundary

When we fit the `LogisticRegression` model, it searches for the optimal weights. However, instead of minimizing Mean Squared Error (MSE), it minimizes **Log Loss** (Binary Cross-Entropy) using an optimization algorithm (like Gradient Descent).

By default, the model sets a **Decision Boundary** at 0.5 (50%). 
* If $\hat{y} \ge 0.5$, it predicts Class 1 (Pass).
* If $\hat{y} < 0.5$, it predicts Class 0 (Fail).

In [ ]:
# Cell 6: Model Instantiation and Fitting

# Instantiate the Logistic Regression model
model = LogisticRegression()

# Fit the model to the training data
model.fit(X_train, y_train)

# Extract the learned parameters
learned_intercept = model.intercept_[0]
learned_slope = model.coef_[0][0]

# Analytically calculate the Decision Boundary
# The boundary is where P = 0.5, which happens when z = 0
# 0 = intercept + slope * x  =>  x = -intercept / slope
decision_boundary = -learned_intercept / learned_slope

print("--- Analytical Model Parameters ---")
print(f"Learned Intercept: {learned_intercept:.4f}")
print(f"Learned Slope:     {learned_slope:.4f}")
print(f"Decision Boundary: {decision_boundary:.2f} hours")
print(f"\nInterpretation: If a student studies more than {decision_boundary:.2f} hours, the model predicts they will pass.")

## Step 3: Visualizing the S-Curve (Probabilities vs. Predictions)

One of the most powerful features of Logistic Regression is `predict_proba()`. Instead of just giving us a hard 1 or 0, it gives us the underlying percentage confidence of its prediction.

Let's draw the mathematically fitted Sigmoid curve over our test data.

In [ ]:
# Cell 8: Extracting Probabilities and Plotting the Curve

# Generate a smooth line of X values from 0 to 10 for plotting the curve
X_smooth = np.linspace(0, 10, 300).reshape(-1, 1)

# Get the underlying probabilities (returns a 2D array: [Prob_Class_0, Prob_Class_1])
# We slice [:, 1] to get just the probability of passing
probabilities = model.predict_proba(X_smooth)[:, 1]

plt.figure(figsize=(10, 6))

# Plot Test Data
plt.scatter(X_test[y_test==0], y_test[y_test==0], edgecolors='k', alpha=0.8, label='Actual Fail')
plt.scatter(X_test[y_test==1], y_test[y_test==1], edgecolors='k', alpha=0.8, label='Actual Pass')

# Plot the Sigmoid Curve
plt.plot(X_smooth, probabilities, color='red', linewidth=3, label='Logistic S-Curve (Probability)')

# Draw the Decision Boundary
plt.axvline(x=decision_boundary, color='gray', linestyle='--', label=f'Decision Boundary ({decision_boundary:.2f}h)')
plt.axhline(y=0.5, color='gray', linestyle=':', alpha=0.5)

plt.title('Logistic Regression: Probability Curve on Test Data', fontsize=14)
plt.xlabel('Hours Studied')
plt.ylabel('Probability of Passing (P)')
plt.legend(loc='upper left')
plt.show()

## Step 4: Classification Evaluation Metrics

Because our targets are categories, we evaluate our model by counting *True Positives*, *True Negatives*, *False Positives*, and *False Negatives*. These make up the **Confusion Matrix**.

From the matrix, we derive several analytical metrics:
* **Accuracy:** Overall percentage of correct predictions.
* **Precision:** When the model predicted "Pass", how often was it right?
* **Recall (Sensitivity):** Out of all the actual "Passes", how many did the model find?

In [ ]:
# Cell 10: Evaluation and Confusion Matrix

# Get the hard predictions (0 or 1) for the test set
y_pred = model.predict(X_test)

# 1. Overall Accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy * 100:.2f}%\n")

# 2. Detailed Classification Report
print("--- Classification Report ---")
print(classification_report(y_test, y_pred, target_names=["Fail (0)", "Pass (1)"]))

# 3. Confusion Matrix Visualization
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Predicted Fail', 'Predicted Pass'],
            yticklabels=['Actual Fail', 'Actual Pass'])
plt.title('Confusion Matrix', fontsize=14)
plt.show()

# Analytical breakdown of the matrix
tn, fp, fn, tp = cm.ravel()
print(f"Analytical Breakdown:")
print(f"- True Negatives (Correctly predicted Fail):  {tn}")
print(f"- True Positives (Correctly predicted Pass):  {tp}")
print(f"- False Positives (Type I Error - Predicted Pass, but Failed): {fp}")
print(f"- False Negatives (Type II Error - Predicted Fail, but Passed): {fn}")

## Conclusion

You have successfully transitioned from continuous predictions to binary classification!

**Key Analytical Takeaways:**
1. **The Link Function:** Logistic Regression is just Linear Regression wrapped in a Sigmoid function to bound the outputs between 0 and 1.
2. **Probability First:** The algorithm fundamentally predicts probabilities (`predict_proba`), and we apply a threshold (usually 0.5) to convert those probabilities into hard classes (`predict`).
3. **Classification Metrics:** MSE and $R^2$ do not apply here. We use the Confusion Matrix, Accuracy, Precision, and Recall to rigorously evaluate the health of a classification model.

This simple but highly interpretable model is the baseline for almost all classification tasks in the real world, from medical diagnoses to financial fraud detection!